<a href="https://colab.research.google.com/github/sanjaliroy/berkeley-homes-wildfire-agent-simulation/blob/main/notebooks/wildfire_transcript_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homeowners Interview Transcript Pre-processing

Course: INFO 290: Fundamentals of Generative AI (Spring 2026)







Creator: Amrita Nambiar

What this notebook covers:
1.   Transcription Cleaning: Filters raw text to isolate homeowner responses.
2.   AI-Driven Extraction: Uses Claude claude-sonnet-4-5 model to transform cleaned transcripts into structured profiles.
3.  Data Export: Saves results to Google Drive as agents_extracted.yaml (profiles) and extraction_debug.jsonl (logs).

Why this matters:


1.   Agent Persona Creation: The extracted profiles form the basis for creating realistic and diverse agent personas in a wildfire mitigation simulation.
2.    Structured Data: Converts unstructured interview data into a structured format that can be easily consumed by simulation models.
3.    Automated Extraction: Leverages Generative AI to automate the extraction process, reducing manual effort and ensuring consistency.



## Install Dependencies

In [1]:
!pip install python-docx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 8.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata, drive
import anthropic
from pathlib import Path

import re
import json
import yaml
from docx import Document
from pathlib import Path
import pandas as pd

## Configuration

In [3]:
# Google Drive folder with interview transcripts
transcripts_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts"

# Output folder
output_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs"

# Output filenames
# Each agent is saved as <agent_id>.yaml in output_folder_path
debug_jsonl_filename = "extraction_debug.jsonl"

# Model for extraction
model = "claude-sonnet-4-5"
max_tokens = 16000

# Interviewer name to filter out from transcripts
interviewer_name = "Amrita"

print("Configuration completed")
print(f"Transcripts folder : {transcripts_folder_path}")
print(f"Output folder      : {output_folder_path}")
print(f"Model              : {model}")


Configuration completed
Transcripts folder : /content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts
Output folder      : /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs
Model              : claude-sonnet-4-5


## Mount Google Drive & Load API Key

In [4]:
# Mount Google Drive
drive.mount("/content/drive")
print("Google Drive mounted")

# Load API key
anthropic_api_key = userdata.get('wildfire_claude')

if anthropic_api_key:
    os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key
    print("API key loaded.")
else:
    print("API key not found. Please add ANTHROPIC_API_KEY to Colab secrets.")

# Verify transcript folder exists
transcript_dir = Path(transcripts_folder_path)
if not transcript_dir.exists():
    print(f"\nTranscript folder not found: {transcripts_folder_path}")
else:
    docx_files = sorted(transcript_dir.glob("*.docx"))
    print(f"\nFound {len(docx_files)} .docx transcript(s):")
    for f in docx_files:
        print(f"   • {f.name}")

Mounted at /content/drive
Google Drive mounted
API key loaded.

Found 11 .docx transcript(s):
   • Beth - Mar 6 at 7_03 PM.docx
   • Charles - Mar 6 at 8_06 PM.docx
   • Edward - Mar 11 at 2_06 PM.docx
   • Isabelle G - Feb 02 2026, 5_00 PM.docx
   • Jennifer - Feb 25 at 1_03 PM.docx
   • Lola - Feb 23 at 5_09 PM.docx
   • Michel_.docx
   • Ross - Feb 25 at 10_21 AM.docx
   • Steven L - Feb 02 2026, 12_00 PM.docx
   • Susan_.docx
   • Yen_.docx


## Helper Functions

In [5]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    # Read all paragraphs from a .docx file and return as plain text
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [6]:
# Filter out interviewer turns
def filter_to_interviewee_only(transcript: str, interviewer_name: str = "Amrita") -> str:
    """
    For timestamped transcripts,strip out all interviewer turns. Return homeowner speech only.
    For unformatted transcripts, return full text unchanged.
    """
    # Detect if transcript uses speaker-timestamp format
    if not re.search(r'\w[\w\s]+:\s+\d{2}:\d{2}', transcript):
        return transcript  # if unformatted, return as it is

    lines = transcript.split('\n')
    filtered = []
    include = True

    for line in lines:
        speaker_match = re.match(r'^([\w][\w\s]+):\s+\d{2}:\d{2}', line)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            include = interviewer_name not in speaker
        if include:
            filtered.append(line)

    return '\n'.join(filtered)


In [7]:
extraction_prompt = """\
You are a research assistant helping build realistic agent personas for a wildfire \
mitigation simulation set in the Berkeley Hills, California.


Below is an interview transcript with a Berkeley Hills resident. Your job is to \
extract a structured profile that will be used to initialise a simulation agent. \
Extract only the profile of the interviewee (Berkeley Hills resident). \
DO NOT include the interviewer's (graduate student at Berkeley) questions or responses in the profile.


TRANSCRIPT:
{transcript}


---


EXTRACTION ORDER — MANDATORY


Extract fields in this order. Each stage constrains the next.


STAGE 1 — held_out_responses
STAGE 2 — memory_seeds
STAGE 3 — seed_narrative


The boundary rule: for each of the five intervention types, identify the interviewee's \
single most representative reflection, what they thought, felt, or believed about the \
situation, and place it in held_out_responses. Everything else they said about that \
event — actions taken, context, conversations, other feelings, belongs in memory_seeds. \
seed_narrative establishes identity and attitude only; it must not repeat episodic detail \
specific actions taken, insurance status, or financial figures, or any information already \
in memory_seeds, and does not reproduce any held-out reflection.\


---


GROUNDING RULE — MANDATORY FOR ALL STAGES


Every field in this extraction — held_out_responses, memory_seeds, and seed_narrative — \
must be grounded directly in the transcript. This means:


- Do not invent facts, events, emotions, or attitudes that the interviewee did not \
  express or clearly imply.
- Do not generalise from one statement to a broader trait or pattern unless the \
  transcript explicitly supports it.
- Do not fill gaps with plausible-sounding detail. If something was not discussed, \
  use null or omit it entirely.
- Before writing any field, ask: "Can I point to a specific moment in the transcript \
  that supports this?" If not, do not include it.


This rule takes precedence over completeness. A shorter, accurate extraction is \
preferable to a fuller one that introduces invented content.


---


STAGE 1: held_out_responses


For each intervention type below, capture the interviewee's single most representative \
reflection — what they thought, felt, or believed about the situation, not what they did. \
ONLY 1 REFLECTION PER INTERVENTION.\
Record this as a direct quote.\
Use null if the topic was not discussed.


 (a) insurance_non_renewal
 (b) defensible_space_inspection
 (c) neighbor_pressure
 (d) firewise_outreach
 (e) resources_assistance


Once extracted, treat each held-out reflection as a quarantine item. \
Stages 2 and 3 must not reproduce it verbatim or as a close paraphrase.


---


STAGE 2: memory_seeds


Memory seeds are discrete episodic memories retrieved dynamically by the simulation. \
They should cover the full range of what the interviewee said:


- Background context for intervention events (insurance, defensible space inspection, \
 neighbor pressure, firewise outreach, resources assistance) \
- Actions taken around intervention events
- Other feelings and attitudes expressed (EXCLUDING the held-out reflection)
- Conversations with neighbors, officials, or contractors
- Observations about the property, neighborhood, insurance or fire risk
- Routine mitigation steps and efforts


Before writing each seed, ask:
 1. "Does this reproduce the exact held-out reflection verbatim or as a close \
    paraphrase?" If yes, omit it and capture the surrounding context or actions \
    without reproducing the specific held-out thought or feeling.
 2. "Is this grounded in something the interviewee explicitly said or clearly \
    implied in the transcript?" If no, do not include it.


Include up to 30 seeds as the transcript supports. Aim for a spread across the \
importance range — do not cluster at high importance. Include importance 1-2 seeds \
only if they capture a distinctly unique personality detail not reflected elsewhere. \
All seeds must be directly grounded in the transcript; do not invent or generalise.


---


STAGE 3: seed_narrative

The seed narrative is the agent's stable identity. It must be 10 sentences maximum.

Before writing each sentence, ask:
 1. "Does this reproduce the exact held-out reflection verbatim or as a close paraphrase?" If yes, omit it.
 2. "Does this reproduce, or duplicate a memory seed?" If yes, omit. Count the sentences in memory_seeds that describe the same trait — if any seed already captures it, do not include it in the narrative.
 3. "Is this grounded in the transcript?" If no, omit.

It must describe behaviours and styles.

DO NOT mention: specific actions, events, secnarios, etc. If a detail of that kind appears in a memory seed or held out reflection, it belongs only there.

Structure in second person ('You are...'):

 (1) Who you are and your living situation — age, occupation, marital status, household, tenure (homeowner or renter), location, risk_zone, how long you've been there, your general character and voice.
 (2) Your fire risk awareness and attitude.
 (3) Your general approach to tasks and community.

Weave these into natural prose without subheadings.
DO NOT reproduce, duplicate or paraphrase the held-out reflection for any intervention type.
Use the same voice and hedging language the person uses in the transcript.
If the transcript contains contradictions, reflect the tension honestly rather than resolving it in favor of one reading.


---

Extract the following and respond ONLY with valid JSON (no markdown, no preamble, \
no trailing text):


{{
 "agent_id": "<firstname_lowercase — e.g. ross, yen, beth>",
 "display_name": "<privacy-preserving pseudonym matching the person's approximate age, \
gender, and cultural background. Never use the interviewee's real name.>",
 "persona": "<3-5 sentence paragraph in third person capturing: who they are, how long \
they have lived there, their attitude toward fire risk, their relationship with the Ember \
ordinance and defensible space requirements, their trust in government and fire \
institutions, and any distinctive personality traits revealed in the interview>",
 "risk_zone": "<high | medium | low — ridge/hilltop=high, mid-hill=medium, flatter/lower=low, or null>",
 "compliance_status": "<compliant | partially_compliant | non_compliant | null>",
 "insurance_status": "<retained | non_renewed | dropped | null>",
 "held_out_responses": {{
   "insurance_non_renewal": {{
     "real_response": "<the interviewee's single most representative reflection — what \
they thought, felt, or believed about the situation. Direct quote or very close \
paraphrase. Null if not discussed.>",
     "context": "<The context for the response or null>"
   }},
   "defensible_space_inspection": {{
     "real_response": "<the interviewee's single most representative reflection — what \
they thought, felt, or believed. Null if not discussed.>",
     "context": "<The context for the response or null>"
   }},
   "neighbor_pressure": {{
     "real_response": "<the interviewee's single most representative reflection — what \
they thought, felt, or believed. Null if not discussed.>",
     "context": "<The context for the response or null>"
   }},
   "firewise_outreach": {{
     "real_response": "<the interviewee's single most representative reflection — what \
they thought, felt, or believed. Null if not discussed.>",
     "context": "<The context for the response or null>"
   }},
   "resources_assistance": {{
     "real_response": "<the interviewee's single most representative reflection — what \
they thought, felt, or believed. Null if not discussed.>",
     "context": "<The context for the response or null>"
   }}
 }},
 "memory_seeds": [
   {{
     "description": "<specific first-person memory grounded in the transcript — past \
tense, concrete detail. May include content from intervention events as long as the \
specific held-out reflection (real_response) is not reproduced verbatim or as a close paraphrase.>",
     "importance": "<integer 1-10: 9-10 = defining life/property event; \
7-8 = significant action or strong emotional reaction; \
5-6 = meaningful conversation or decision about mitigation; \
3-4 = routine observation or minor step; \
1-2 = passing remark or peripheral detail>",
     "type": "<observation | reflection | conversation>",
     "source": "<direct_experience | secondhand_community | secondhand_media | institutional>"
   }}
 ],
 "seed_narrative": "<Narrative in second person ('You are {{display_name}}...') \
structured across the dimensions described in Stage 3. Establish identity and attitude \
of interviewee only. DO NOT include examples of situations in memory_seeds or held_out_responses real_response \
do not reproduce, paraphrase or duplicate any held-out reflection, \
do not reproduce, paraphrase or duplicate episodic detail already in memory_seeds, \
make sure is grounded entirely in the transcript. Do not invent. \
End the narrative with the following text verbatim: 'You will now respond to situations \
as {{display_name}}. Focus only on what you would actually do in direct response to the \
situation presented. Do not add unrelated action items or narrative conclusions. Stay \
practical, not emotional or literary. Stay in their voice. Respond in first person only. \
Do not narrate actions in third person, do not use stage directions or asterisks, do not \
use headers, bold text, horizontal rules, em-dashes, or formatted sections. Write in \
plain short sentences. Keep your response concise.'>",
"key_concerns": [
   "<specific worry or tension the homeowner expressed>"
 ],
 "notable_quotes": [
   "<short verbatim quote under 20 words>",
   "<another quote>"
 ]
}}
"""

In [8]:
# Call Claude to extract profile
def extract_agent_profile(client: anthropic.Anthropic, transcript: str, name: str) -> tuple:
    # Send transcript to Claude and return (parsed_dict, raw_text)

    prompt = extraction_prompt.format(transcript=transcript)

    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = message.content[0].text.strip()

    # Strip markdown fences if the model adds them despite instructions
    raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
    raw_text = re.sub(r'\s*```$', '', raw_text).strip()

    try:
        profile = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {name}: {e}")
        profile = {"agent_id": name.lower(), "parse_error": str(e), "raw": raw_text}

    return profile, raw_text

In [9]:
# Convert profile to agents.yaml format
def profile_to_agent_entry(profile: dict) -> dict:
    # Reshape extracted profile into the agents.yaml schema
    if "parse_error" in profile:
        return {"id": profile.get("agent_id", "unknown"), "error": profile["parse_error"]}

    return {
        "id":                      profile.get("agent_id"),
        "display_name":            profile.get("display_name"),
        "persona":                 profile.get("persona", ""),
        "risk_zone":               profile.get("risk_zone"),
        "compliance_status":       profile.get("compliance_status"),
        "insurance_status":        profile.get("insurance_status"),
        "held_out_responses":      profile.get("held_out_responses", {}),
        "memory_seeds":            profile.get("memory_seeds", []),
        "seed_narrative":          profile.get("seed_narrative", ""),
        "key_concerns":            profile.get("key_concerns", []),
        "notable_quotes":          profile.get("notable_quotes", []),
    }


print("Helper functions defined")

Helper functions defined


## Run Extraction (All Transcripts)

In [10]:
# Setup
claude_client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY from environment

transcript_dir = Path(transcripts_folder_path)
output_dir = Path(output_folder_path)
output_dir.mkdir(parents=True, exist_ok=True)

docx_files = sorted(transcript_dir.glob("*.docx"))
print(f"Processing {len(docx_files)} transcripts...\n")
print("=" * 60)

agents = []
debug_entries = []
errors = []

# Main extraction loop
for i, docx_path in enumerate(docx_files, 1):
    # Derive a short name from the filename
    stem = docx_path.stem
    name = stem.split(" - ")[0].split("_")[0].strip()

    print(f"[{i}/{len(docx_files)}] {name}  ({docx_path.name})")

    try:
        # Extract raw text
        full_text = extract_text_from_docx(docx_path)
        word_count = len(full_text.split())
        print(f"         {word_count:,} words extracted")

        # Filter to homeowner voice only
        homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
        filtered_wc = len(homeowner_text.split())
        if filtered_wc < word_count:
            print(f"         → filtered to {filtered_wc:,} words (interviewer turns removed)")

        # Call Claude
        profile, raw_response = extract_agent_profile(claude_client, homeowner_text, name)

        # Convert to agents.yaml format
        agent_entry = profile_to_agent_entry(profile)
        agents.append(agent_entry)

        # --- Save individual YAML file named by agent_id ---
        agent_id = agent_entry.get("id") or name.lower()
        agent_yaml_path = output_dir / f"{agent_id}.yaml"
        with open(agent_yaml_path, "w", encoding="utf-8") as f:
            yaml.dump(
                agent_entry,
                f,
                default_flow_style=False,
                allow_unicode=True,
                sort_keys=False,
            )
        print(f"         Saved → {agent_yaml_path.name}")

        # Save debug info
        debug_entries.append({
            "file": docx_path.name,
            "name": name,
            "word_count_full": word_count,
            "word_count_filtered": filtered_wc,
            "raw_llm_response": raw_response,
            "parsed_profile": profile
        })

        # Print summary row
        print(f"         agent_id={agent_entry.get('id')} | "
              f"risk={agent_entry.get('risk_zone')} | "
              f"compliance={agent_entry.get('compliance_status')}")

    except Exception as e:
        print(f"         ERROR: {e}")
        errors.append({"file": docx_path.name, "error": str(e)})

    print()

print("=" * 60)
print(f"Done. {len(agents)} agents extracted, {len(errors)} errors.")


Processing 11 transcripts...

[1/11] Beth  (Beth - Mar 6 at 7_03 PM.docx)
         6,832 words extracted
         Saved → beth.yaml
         agent_id=beth | risk=high | compliance=partially_compliant

[2/11] Charles  (Charles - Mar 6 at 8_06 PM.docx)
         6,236 words extracted
         Saved → charles.yaml
         agent_id=charles | risk=high | compliance=partially_compliant

[3/11] Edward  (Edward - Mar 11 at 2_06 PM.docx)
         1,195 words extracted
         Saved → edward.yaml
         agent_id=edward | risk=high | compliance=compliant

[4/11] Isabelle G  (Isabelle G - Feb 02 2026, 5_00 PM.docx)
         6,935 words extracted
         → filtered to 4,390 words (interviewer turns removed)
         Saved → isabelle.yaml
         agent_id=isabelle | risk=high | compliance=compliant

[5/11] Jennifer  (Jennifer - Feb 25 at 1_03 PM.docx)
         4,352 words extracted
         Saved → jennifer.yaml
         agent_id=jennifer | risk=high | compliance=partially_compliant

[6/11] Lol

## Save Outputs to Google Drive

In [11]:
# Save extraction_debug.jsonl
debug_jsonl_path = output_dir / debug_jsonl_filename

with open(debug_jsonl_path, "w", encoding="utf-8") as f:
    for entry in debug_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved debug log → {debug_jsonl_path}")
print(f"\nPer-agent YAML files saved to: {output_dir}")
for a in agents:
    aid = a.get('id', 'unknown')
    print(f"   • {aid}.yaml")

if errors:
    print(f"\n{len(errors)} file(s) failed:")
    for e in errors:
        print(f"   • {e['file']}: {e['error']}")


Saved debug log → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/extraction_debug.jsonl

Per-agent YAML files saved to: /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs
   • beth.yaml
   • charles.yaml
   • edward.yaml
   • isabelle.yaml
   • jennifer.yaml
   • lola.yaml
   • michelle.yaml
   • ross.yaml
   • steven.yaml
   • susan.yaml
   • yen.yaml


Summary Table

In [12]:
import pandas as pd

rows = []
for a in agents:
    rows.append({
        "ID":                      a.get("id", None),
        "Display Name":            a.get("display_name", None),
        "Persona":                 a.get("persona", None),
        "Risk Zone":               a.get("risk_zone"),
        "Compliance Status":       a.get("compliance_status"),
        "Insurance Status":        a.get("insurance_status"),
        "Held Out Responses":      a.get("held_out_responses", {}),
        "Memory Seeds":            a.get("memory_seeds", []),
        "Seed Narrative":          a.get("seed_narrative"),
        "Key Concerns":            a.get("key_concerns", []),
        "Notable Quotes":          a.get("notable_quotes", []),
    })

df = pd.DataFrame(rows)
print("\nEXTRACTED AGENT SUMMARY")
display(df)

# Distribution checks — useful for ensuring variance across agent pool
print("\nDISTRIBUTIONS")
for col in ["ID", "Display Name", "Persona", "Risk Zone",
            "Compliance Status", "Insurance Status",
             "Held Out Responses","Memory Seeds","Seed Narrative","Key Concerns", "Notable Quotes"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())


EXTRACTED AGENT SUMMARY


,ID,Display Name,Persona,Risk Zone,Compliance Status,Insurance Status,Held Out Responses,Memory Seeds,Seed Narrative,Key Concerns,Notable Quotes
0,beth,Linda,Linda is an older working lawyer who has lived...,high,partially_compliant,retained,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I live right at the top of t...,"You are Linda, an older working lawyer who has...","[Inconsistent guidance from young, inexperienc...",[They were basically sending interns out to do...
1,charles,Robert,Robert is a longtime Berkeley Hills homeowner ...,high,partially_compliant,non_renewed,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I had not seen an inspection...,"You are Robert, a retired risk consultant in y...",[Insurance non-renewal leaving homeowners with...,"[Left holes in the bag literally., You gotta d..."
2,edward,Walter,Walter is a homeowner over 65 living on a larg...,high,compliant,None,{'insurance_non_renewal': {'real_response': No...,[{'description': 'I have a big yard on the eas...,"You are Walter, a homeowner over 65 living on ...",[Regulatory contradictions between fire safety...,"[A tree is anything that you call a tree., Jun..."
3,isabelle,Claire,Claire is a Berkeley Hills homeowner who lives...,high,compliant,retained,{'insurance_non_renewal': {'real_response': 'A...,[{'description': 'I started wildfire mitigatio...,"You are Claire, a longtime Berkeley Hills home...",[Living across the street from Wildcat Canyon ...,"[that really kind of shocked us, Except for al..."
4,jennifer,Laura,Laura is a Berkeley Hills homeowner who lives ...,high,partially_compliant,retained,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I started working on our hom...,"You are Laura, a Berkeley Hills homeowner livi...","[Insurance feels capricious and unfair, underm...","[It's very unfair, and it feels capricious., N..."
5,lola,Margaret,Margaret is an older Berkeley Hills resident l...,high,compliant,non_renewed,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I went on Nextdoor and found...,"You are Margaret, an older Berkeley Hills resi...",[Neighbors' 47 eucalyptus trees dropping highl...,[I'm also sentimental about not burning to dea...
6,michelle,Robert,Robert is a long-time Berkeley Hills resident ...,high,compliant,retained,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I escaped my house during a ...,"You are Robert, a 28-year resident of Creston ...",[Insurance companies will enforce stricter req...,[It's the same exact phenomenon that you see w...
7,ross,Marcus,Marcus is a financially secure homeowner who h...,high,partially_compliant,retained,{'insurance_non_renewal': {'real_response': 'I...,[{'description': 'I moved into this house abou...,You are Marcus. You are a financially comforta...,[Difficulty interpreting defensible space requ...,[I've got the will. I got the time. I got the ...
8,steven,David Chen,David Chen is a homeowner in the Berkeley Hill...,high,compliant,retained,{'insurance_non_renewal': {'real_response': 'S...,[{'description': 'My home was built in 2011 an...,"You are David Chen, a homeowner in the Berkele...",[Whether insurance companies will continue to ...,[We're unusual in that our home is like the on...
9,susan,Susan Chen,Susan Chen is a retired former Berkeley City C...,high,None,None,{'insurance_non_renewal': {'real_response': 'S...,[{'description': 'I represented the Northeast ...,"You are Susan Chen, a retired former Berkeley ...",[Insurance companies canceling policies and ho...,"[social media is a swamp, It's kind of like va..."



DISTRIBUTIONS

ID:
ID
beth        1
charles     1
edward      1
isabelle    1
jennifer    1
lola        1
michelle    1
ross        1
steven      1
susan       1
yen         1

Display Name:
Display Name
Robert        2
Linda         1
Walter        1
Claire        1
Laura         1
Margaret      1
Marcus        1
David Chen    1
Susan Chen    1
Linda Chen    1

Persona:
Persona
Linda is an older working lawyer who has lived in the Berkeley Hills for many years. She has an extensive garden that she is deeply invested in and has evacuated once before though no fire occurred. She approaches fire mitigation compliance pragmatically but with frustration about inconsistent messaging from young, inexperienced inspectors and the lack of action by regional authorities to create firebreaks in the parks. She is relatively compliant and willing to spend significant money on mitigation, but questions why homeowners bear the entire burden when the city and regional parks control the most dangerous

##Inspect Individual Agent Profile

In [13]:
# Change this to the agent_id you want to inspect
inspect_agent = "beth"

match = next((a for a in agents if a.get("id") == inspect_agent), None)

if match is None:
    print(f"Agent '{inspect_agent}' not found. Available: {[a.get('id') for a in agents]}")
else:
    print(f"=== {match['display_name'].upper()} ===")
    print(f"\nPERSONA:")
    print(f"  {match['persona']}")

    print(f"\nKEY ATTRIBUTES:")
    print(f"  risk_zone            : {match.get('risk_zone')}")
    print(f"  compliance_status    : {match.get('compliance_status')}")
    print(f"  insurance_status     : {match.get('insurance_status')}")

    print(f"\nHELD OUT RESPONSES:")
    held_out = match.get('held_out_responses', {})
    if held_out.get('insurance_non_renewal'):
        print(f"  Insurance Non-Renewal: ")
        print(f"    Real Response: {held_out['insurance_non_renewal'].get('real_response')}")
        print(f"    Context: {held_out['insurance_non_renewal'].get('context')}")
    if held_out.get('defensible_space_inspection'):
        print(f"  Defensible Space Inspection: ")
        print(f"    Real Response: {held_out['defensible_space_inspection'].get('real_response')}")
        print(f"    Context: {held_out['defensible_space_inspection'].get('context')}")
    if held_out.get('neighbor_pressure'):
        print(f"  Neighbor Pressure: ")
        print(f"    Real Response: {held_out['neighbor_pressure'].get('real_response')}")
        print(f"    Context: {held_out['neighbor_pressure'].get('context')}")
    if held_out.get('firewise_outreach'):
        print(f"  Firewise Outreach: ")
        print(f"    Real Response: {held_out['firewise_outreach'].get('real_response')}")
        print(f"    Context: {held_out['firewise_outreach'].get('context')}")
    if held_out.get('resources_assistance'):
        print(f"  Resources Assistance: ")
        print(f"    Real Response: {held_out['resources_assistance'].get('real_response')}")
        print(f"    Context: {held_out['resources_assistance'].get('context')}")

    print(f"\nSEED MEMORIES:")
    for i, mem in enumerate(match.get('memory_seeds', []), 1):
        print(f"  {i}. {mem.get('description', 'N/A')}")

    print(f"\nSEED NARRATIVE:")
    print(f"  {match.get('seed_narrative')}")


    print(f"\nKEY CONCERNS:")
    for concern in match.get('key_concerns', []):
        print(f"  • {concern}")

    print(f"\nNOTABLE QUOTES:")
    for quote in match.get('notable_quotes', []):
        print(f'  "{quote}"')


=== LINDA ===

PERSONA:
  Linda is an older working lawyer who has lived in the Berkeley Hills for many years. She has an extensive garden that she is deeply invested in and has evacuated once before though no fire occurred. She approaches fire mitigation compliance pragmatically but with frustration about inconsistent messaging from young, inexperienced inspectors and the lack of action by regional authorities to create firebreaks in the parks. She is relatively compliant and willing to spend significant money on mitigation, but questions why homeowners bear the entire burden when the city and regional parks control the most dangerous areas. She values her time over money at this stage of life and relies heavily on neighbor recommendations rather than getting multiple contractor estimates.

KEY ATTRIBUTES:
  risk_zone            : high
  compliance_status    : partially_compliant
  insurance_status     : retained

HELD OUT RESPONSES:
  Insurance Non-Renewal: 
    Real Response: I'm ve

## Preview Individual Agent YAML Output


In [14]:
# Preview the first agent's individual YAML file
if agents:
    first = agents[0]
    aid = first.get('id', 'unknown')
    print(f"--- {aid}.yaml ---")
    print(yaml.dump(first, default_flow_style=False, allow_unicode=True, sort_keys=False))
else:
    print("No agents extracted yet.")


--- beth.yaml ---
id: beth
display_name: Linda
persona: Linda is an older working lawyer who has lived in the Berkeley Hills for
  many years. She has an extensive garden that she is deeply invested in and has evacuated
  once before though no fire occurred. She approaches fire mitigation compliance pragmatically
  but with frustration about inconsistent messaging from young, inexperienced inspectors
  and the lack of action by regional authorities to create firebreaks in the parks.
  She is relatively compliant and willing to spend significant money on mitigation,
  but questions why homeowners bear the entire burden when the city and regional parks
  control the most dangerous areas. She values her time over money at this stage of
  life and relies heavily on neighbor recommendations rather than getting multiple
  contractor estimates.
risk_zone: high
compliance_status: partially_compliant
insurance_status: retained
held_out_responses:
  insurance_non_renewal:
    real_response: I'm 

## Re-run a Single Agent

Use this if one extraction failed or produced a poor result. It re-runs just that one file without re-processing the others.

In [15]:
# Set to the filename you want to re-run (just the filename, not the full path)
rerun_file = "Ross - Feb 25 at 10_21 AM.docx"

rerun_path = transcript_dir / rerun_file

if not rerun_path.exists():
    print(f"File not found: {rerun_path}")
else:
    name = rerun_file.split(" - ")[0].split("_")[0].strip()
    print(f"Re-running extraction for: {name}")

    full_text = extract_text_from_docx(rerun_path)
    homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
    profile, raw_response = extract_agent_profile(claude_client, homeowner_text, name)
    agent_entry = profile_to_agent_entry(profile)

    # Replace in agents list if already exists, otherwise append
    existing_ids = [a.get("id") for a in agents]
    if agent_entry.get("id") in existing_ids:
        idx = existing_ids.index(agent_entry.get("id"))
        agents[idx] = agent_entry
        print(f"Replaced existing entry for '{agent_entry.get('id')}'")
    else:
        agents.append(agent_entry)
        print(f"Added new entry for '{agent_entry.get('id')}'")

    print(f"\nResult:")
    print(json.dumps(agent_entry, indent=2))

    # Save individual YAML for the re-run agent
    agent_id = agent_entry.get("id") or name.lower()
    agent_yaml_path = output_dir / f"{agent_id}.yaml"
    with open(agent_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(agent_entry, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print(f"\nSaved → {agent_yaml_path}")


Re-running extraction for: Ross
Replaced existing entry for 'ross'

Result:
{
  "id": "ross",
  "display_name": "Marcus",
  "persona": "Marcus is a Berkeley Hills homeowner who has lived in his house for about five years on Wildcat Canyon Road, right next to Tilden Park. He is financially capable and highly motivated to comply with fire safety requirements, but finds the Ember ordinance confusing and difficult to interpret. He has experienced significant frustration with inconsistent inspection feedback and the lack of clear, personalized guidance from fire officials. Marcus is willing to invest substantial time and money into fire hardening his property, but feels hampered by ambiguity in regulations and insufficient direct communication with authorities. He is part of a neighborhood firewise community that has been affected by disinformation campaigns from residents opposed to the ordinance.",
  "risk_zone": "high",
  "compliance_status": "partially_compliant",
  "insurance_status": 